In [ ]:
# io
import pathlib
import functools 
import datetime

# analysis
import scipy
import numpy
import pandas

# plotting
import matplotlib.pyplot as plt

# pmana
import pmana.utils
import pmana.purity

EXPERIMENT = f"NP02"

In [ ]:
# configure analysis
ANALYSIS_CONFIGURATION = pmana.purity.config.DEFAULT_ANALYSIS_CONFIGURATION

ANALYSIS_CONFIGURATION['ExtractKwargs'] = dict(
  IS_CSV = False,
  COL_NAMES = ['BinCenter', 'Population'],
  DELIMITER = ','
)

# channel mapping
ANALYSIS_CONFIGURATION['InnerLongChannel']  = 1
ANALYSIS_CONFIGURATION['OuterLongChannel']  = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 2
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 0

# compton normalization search
ANALYSIS_CONFIGURATION['ShortComptonSearchLimits']        = (0.6, 0.9)
ANALYSIS_CONFIGURATION['ShortMinComptonSearchLowLimit']   = 0.4
ANALYSIS_CONFIGURATION['LongComptonSearchLimits']         = (0.5, 0.9)
ANALYSIS_CONFIGURATION['LongMinComptonSearchLowLimit']    = 0.4
ANALYSIS_CONFIGURATION['ComptonMode']                     = 'rising'

# limits to search and fit the IC peaks, after outer-to-inner subtraction
ANALYSIS_CONFIGURATION['LongICPeakSearchLimits']  = (0.7, 1)  # long Bi-PrM has a significant tail...
ANALYSIS_CONFIGURATION['ShortICPeakSearchLimits'] = (0.8, 1)  # short Bi-PrM has a significant test pulse...

In [ ]:
# configure analysis
TESTPULSE_CONFIGURATION = pmana.purity.config.CSV_TESTPULSE_CONFIGURATION

# channel mapping
ANALYSIS_CONFIGURATION['InnerLongChannel']  = 1
ANALYSIS_CONFIGURATION['OuterLongChannel']  = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 0
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 2

# cut on the spectrum to isolate the test pulses
TESTPULSE_CONFIGURATION['MaskTestPulse']    = True
TESTPULSE_CONFIGURATION['LongTestPulseLowLim']  = 1
TESTPULSE_CONFIGURATION['LongTestPulseHighLim'] = 1.35
TESTPULSE_CONFIGURATION['ShortTestPulseLowLim']  = 0.65
TESTPULSE_CONFIGURATION['ShortTestPulseHighLim'] = 0.9

# handle the background nicely...
TESTPULSE_CONFIGURATION['TPFitter'] = pmana.utils.fitting.GausPlusLine

#### 1. Temperature calibration

##### 1.1 Temperatures

Get the temperatures from the NP02 slow controls page. 
The page can be quirky, but you should be able to download ambient temperatures by hand.

In [ ]:
PATH_TEMPERATURES = f"../../data/cern/NP02_Apr_Oct_Jan.csv"

# get temperature mapping
Temperatures = pmana.utils.io.ExtractTemperatureMonitoring(
    PATH_TEMPERATURES,
    IsPadova = False,
    delimiter = "\t",
    skiprows = 1,
    # usecols = [0, 1],
    names = ["Date", "T"]
)

# some more formatting...
Temperatures['Date'] = pandas.to_datetime(
    Temperatures['Date'],
    unit='ms',
    utc=True
)
Temperatures = Temperatures.drop_duplicates()
Temperatures = Temperatures.sort_values(by=['Date'])

##### 1.2 Find a stable-lifetime timeframe

Using the data from the UV-PrM, find a timeframe where the lifetime was stable.

In [ ]:
UV_PURITY_DATA = "../../data/cern/prm_Top_lifetime_data_Gain_InducedCurrent_TPC.csv"

UVData = pandas.read_csv(
  UV_PURITY_DATA,
  delimiter = ',',
  names = ['Date', 'PM', 'Center', 'High', 'Low'],
  skiprows=1 
)

# some more formatting...
UVData['Date'] = pandas.to_datetime(
    UVData['Date']
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.75*1), nrows=1, layout='tight')

ax.plot(UVData.Date, UVData.Low, lw=1, c='gray')
ax.plot(UVData.Date, UVData.Center, lw=1, c='black', label='UV-PrM')
ax.plot(UVData.Date, UVData.High, lw=1, c='gray')
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Lifetime [ms]')

# highlight windows
ax.axvspan(datetime.datetime(2025, 1, 13), datetime.datetime(2025, 1, 15), zorder=-3, ec='C0', fc='None', hatch='////', label='Long $\\tau_\\mathrm{max}$') # long PrM
ax.axvspan(datetime.datetime(2025, 5, 28), datetime.datetime(2025, 5, 30), zorder=-3, ec='C1', fc='None', hatch='\\\\\\\\', label='Short $\\tau_\\mathrm{max}$') # short PrM
ax.axvspan(datetime.datetime(2025, 5, 23), datetime.datetime(2025, 5, 30), zorder=-3, ec='C2', fc='None', hatch='////', label='Stable $\\tau$')

# gfx
ax.legend(frameon=True, fancybox=False, handlelength=1, columnspacing=0.5, ncol=3, fontsize=10, loc='upper center')
ax.tick_params(axis='x', labelrotation=30)
ax.set_xlim(min(UVData['Date']), max(UVData['Date']))
# ax.set_xlim(datetime.datetime(2025,12,1), datetime.datetime(2025,12,20))
ax.set_ylim(0, 45)
ax.grid(alpha=0.5)

# temperature
a2 = ax.twinx()
a2.set_zorder(ax.get_zorder() - 1) # this goes on the background
ax.patch.set_visible(False)
a2.plot(Temperatures['Date'], Temperatures['T'], lw=1.5, alpha=0.2, c='red', label='Temperature')
pmana.utils.plotting.UpdateMatplotlibStyle(a2, None, 'Temperature [°C]')
a2.get_legend().remove()

fig.suptitle(f'{EXPERIMENT} data (UV-PrM)', fontsize=16)

ax.set_xlim(datetime.datetime(2025, 1, 1), datetime.datetime(2025, 6, 1))

plt.show()

##### 1.3 Extract the IC peaks in this timeframe

Extract the IC peaks in the timeframe where the lifetime was high and stable.
In this way, any systematic effect on the IC peak does not depend on the LAr purity.

In [ ]:
CAMPAIGN = 'cern'
YEAR = '2025'
MONTH = 'May'
MEASUREMENT = f'24/04/20'
FILE_PATH = f"../../data/{CAMPAIGN}/{YEAR}_{MONTH}/{MEASUREMENT}"
PM_TAG = 'Long'

Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = False,
    COL_NAMES = ['BinCenter', 'Population'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

fig, ax = plt.subplots(figsize=(6,3))

# plot the spectra
CHs = [1, 2, 3, 4]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
    )
    
# perform IC peak analysis
print(f"Analyzing the {PM_TAG} PrM.")
pars, errs, scale, CE, pars_CE, xIC, IC = pmana.purity.ana.ExtractICPeak(
  FILE_PATH,
  Timestamp = None,
  PM_TAG = PM_TAG,
  DEBUG_MODE = True,
  ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION,
)
ax.axvline(CE, lw=0.75, c='red', ls='--', label='Compton edge')

# debug
print(f"{PM_TAG} Scaling factor: ", scale)
print(f"{PM_TAG} Compton edge: ", CE)
print(f"{PM_TAG} Compton edge fit pars: ", pars_CE)

# Compton Edge from fit
# x_CE = numpy.linspace(CE - ANALYSIS_CONFIGURATION['ComptonEdgeFitLeftLimit'], CE + 0.2, 100)
# ax.plot(x_CE, pmana.utils.fitting.Gaus(x_CE, *pars_CE), c='C7', lw=1, ls='-.', label='CE fit')

# extracted IC peak
ax.step(
    xIC, IC,
    where = 'mid',
    c = 'black',
    lw = 1,
    label = 'IC'
)

# triple Gaussian plot
ax.plot(xIC, pmana.utils.fitting.TripleGaus(xIC, *pars), c='red', lw=1, label='triple Gaus fit')
ax.plot(xIC, pars[0] * numpy.exp(- (xIC - pars[1])**2 / (2 * pars[2]**2)), c='C4', ls='--', lw=1)
ax.plot(xIC, pars[0] * 0.262 * numpy.exp(- (xIC - pars[1] * 1.0747)**2 / (2 * pars[2]**2)), c='C5', ls='--', lw=1)
ax.plot(xIC, pars[0] * 0.077 * numpy.exp(- (xIC - pars[1] * 1.0747)**2 / (2 * pars[2]**2)), c='C7', ls='--', lw=1)
ax.axvline(pars[1], c='C4', lw=1, label='μ(IC)')
ax.axvspan(pars[1]-errs[1], pars[1]+errs[1], fc='C4', alpha=0.2)

# gfx
ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Pulse height [V]",
    "Counts [#]"
)
ax.legend(handlelength=1)
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT.replace('_','-')}")

plt.show()

In [ ]:
# analyze campaign
CAMPAIGN = f"cern"
TIME_DIR = f"2025_May"
PATH_CAMPAIGN = f"../../data/{CAMPAIGN}/{TIME_DIR}"
DATE_LOW  = datetime.datetime(2025, 5, 23)
DATE_HIGH = datetime.datetime(2025, 5, 30)

### long PrM
Output = pmana.utils.iterators.IterateCERN(
  PATH_CAMPAIGN,  ###< path to data
  functools.partial(
      pmana.purity.ana.ExtractICPeak, 
      PM_TAG = 'Long',
      ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION,
  ),  ###< analyzing module, changing some options 
  YEAR = 2025,
  MONTH = 5,
  START_FROM  = DATE_LOW,
  END_AT      = DATE_HIGH
)

# convert top DataFrame and re-format the dataframe
Output = pandas.DataFrame(Output)
Output.columns = [
  "LongIC_Peak", "LongIC_PeakErr", "LongIC_Scale",
  "Date"
]

### short PrM
Output_Short = pmana.utils.iterators.IterateCERN(
  PATH_CAMPAIGN,  ###< path to data
  # pmana.utils.anatestdata.AnalyzeMeasurement, 
  functools.partial(
      pmana.purity.ana.ExtractICPeak, 
      PM_TAG = 'Short',
      ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION,
  ),  ###< analyzing module, changing some options 
  YEAR = 2025,
  MONTH = 5,
  START_FROM  = DATE_LOW,
  END_AT      = DATE_HIGH
)

# convert top DataFrame and re-format the dataframe
Output_Short = pandas.DataFrame(Output_Short)
Output_Short.columns = [
  "ShortIC_Peak", "ShortIC_PeakErr", "ShortIC_Scale",
  "Date"
]

### merge
# Output = pandas.merge(Output, Output_Short, on = 'Date', how='inner')
Output = pandas.merge(Output_Short, Output, on = 'Date', how='left')

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3), nrows=1, layout='tight')

ax.scatter(
  Output['Date'], Output['ShortIC_Peak'], 
  c='C0', label='Short Bi-PrM',
  ls='', marker='.'
)

ax.scatter(
  Output['Date'], Output['LongIC_Peak'], 
  c='C1', label='Long Bi-PrM',
  ls='', marker='.'
)

# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'IC Peak [V]')
ax.legend(handlelength=1, loc='lower right', fancybox=False)
ax.tick_params(axis='x', labelrotation=30)
ax.grid(which='major', alpha=0.5)
ax.grid(which='minor', ls='--', alpha=0.3)

ax.set_ylim(0.67, 0.85)

ax.set_title(f'CERN-{EXPERIMENT} data', fontsize=16)

plt.show()

##### 1.4 Correlate IC peaks and temperatures

In [ ]:
# Interpolate temperatures to generalize
Temperatures['Date_s'] = Temperatures["Date"].astype("int64") / 1e9 ###< seconds from epoch, 1970-01-01
Output['Date_s'] = Output["Date"].astype("int64") / 1e9

TempVsTime = scipy.interpolate.interp1d(
  Temperatures['Date_s'],
  Temperatures['T'],
  kind = 'linear'
)
Output['T'] = TempVsTime(Output['Date_s'])

In [ ]:
fig, axes = plt.subplots(figsize=(4*2, 4), ncols=2, layout='tight')

LinearCalibration = []
QuadraticCalibration = []

CHANNELS = ['LongIC_Peak', 'ShortIC_Peak']

for i, ax in enumerate(axes):

        # scatter
        ax.scatter(Output['T'], Output[CHANNELS[i]], marker = '.', c='black', label=f'>{DATE_LOW.strftime('%Y-%m-%d')}\n>{DATE_HIGH.strftime('%Y-%m-%d')}')

        # fits
        x = Output['T'].values
        y = Output[CHANNELS[i]].values

        ### linear
        # december
        MaskFit = (x < 100)
        # # january
        # MaskFit = (x < 16.5)
        par, cov = scipy.optimize.curve_fit(pmana.utils.fitting.Line, x[MaskFit], y[MaskFit]); LinearCalibration.append(par)
        domain = numpy.linspace(min(x), max(x), 100)
        ax.plot(domain, pmana.utils.fitting.Line(domain, *par), c='red', lw=1.5, 
                label=f'linear (>17°C)\n{par[0]:.3f} V/°C, {par[1]:.3f} V')
        err_lin = pmana.utils.fitting.LineErrorBand(domain, cov)
        ax.fill_between(domain, pmana.utils.fitting.Line(domain, *par) - err_lin, pmana.utils.fitting.Line(domain, *par) + err_lin, color='red', alpha=0.1)

        pmana.utils.plotting.UpdateMatplotlibStyle(ax, f'Temperature [°C]', f'{CHANNELS[i]} [V]')
        ax.set_ylim(ax.get_ylim()[0], ax.get_ylim()[1]+0.005)
        ax.legend(loc='upper center', fancybox=False, handlelength=1)